#### Task 1: Simple vector embedding generation

**Objective:**
Generate vector embeddings from text data.

**Task Description:**

- load embedding model into ollama:
    1. Attach ollama container shell
    2. Run command in container `ollama pull pull bge-m3` or send an api call `requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False}` via the requests library to download embedding model.
- embed simple text queries

How to select the right embedding model: [MTEB - Massive Text Embedding Benchmark](https://huggingface.co/blog/mteb)

**Useful links:**

- [Ollama REST API](https://www.postman.com/postman-student-programs/ollama-api/documentation/suc47x8/ollama-rest-api)
- [Langchain Ollama Embeddings](https://python.langchain.com/docs/integrations/text_embedding/ollama/)
- [Langchain Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/)


In [23]:
import requests

In [24]:
# Download embedding model into ollama container
requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False})

<Response [200]>

In [25]:
# List all available models in the ollama container
# Note: /api/ps shows only running models, /api/tags shows all installed models
response = requests.get("http://localhost:11434/api/tags")
response.json()

{'models': [{'name': 'bge-m3:latest',
   'model': 'bge-m3:latest',
   'modified_at': '2026-04-23T10:02:27.500261532Z',
   'size': 1157672605,
   'digest': '7907646426070047a77226ac3e684fbbe8410524f7b4a74d02837e43f2146bab',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'bert',
    'families': ['bert'],
    'parameter_size': '566.70M',
    'quantization_level': 'F16'}},
  {'name': 'qwen3.5:2b',
   'model': 'qwen3.5:2b',
   'modified_at': '2026-04-09T08:53:56.81333625Z',
   'size': 2741192820,
   'digest': '324d162be6ca5629ae4517c8710434d0bd2d665bc94dbad46e9af8fbf8a2f0df',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'qwen35',
    'families': ['qwen35'],
    'parameter_size': '2.3B',
    'quantization_level': 'Q8_0'}}]}

In [26]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

In [27]:
text = "This is a test document."

# ADD HERE YOUR CODE
# Perform vector search
query_vector = embedding_model.embed_query(text)

print(f"Embedding vector length: {len(query_vector)}")
print(query_vector[:10])

Embedding vector length: 1024
[-0.050007477, 0.021129724, -0.057080247, -0.005526018, -0.0062946486, -0.028408367, 0.0451794, 0.042706486, -0.00021894443, 0.013056353]


#### Task 2: Generate embedding vectors with custom dataset

**Objective:**
Load custom dataset, preprocess it and generate vector embeddings.

**Task Description:**

- load pdf document "AI_Book.pdf" via langchain document loader: ``PyPDFLoader``
- use RecursiveCharacterTextSplitter to split documents into chunks
- generate embeddings for single documents

**RecursiveCharacterTextSplitter:**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

**Useful links:**

- [Langchain PyPDFLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/)
- [Langchain RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)


In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import math

pdf_doc = "./AI_Book.pdf"

# Create pdf data loader
# ADD HERE YOUR CODE
loader = PyPDFLoader(pdf_doc)

# Load and split documents in chunks
# ADD HERE YOUR CODE
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pages_chunked = loader.load_and_split(text_splitter)

print(pages_chunked[0])

# Function to clean text by removing invalid unicode characters, including surrogate pairs
def clean_text(text):
    # Remove surrogate pairs
    text = re.sub(r'[\ud800-\udfff]', '', text)
    # Keep only alphanumeric characters and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove 'nan' (case insensitive)
    text = re.sub(r'\bnan\b', '', text, flags=re.IGNORECASE)
    return text

pages_chunked_cleaned = []

for doc in pages_chunked:
    # Text bereinigen und Ränder säubern
    cleaned_text = clean_text(doc.page_content).strip()
    
    # Nur Chunks hinzufügen, die mindestens 10 Zeichen lang sind.
    # Sehr kurze Chunks (z.B. nur "." oder ein einzelnes Wort) verursachen oft NaN-Fehler.
    if len(cleaned_text) > 10 and 'nan' not in cleaned_text.lower(): 
        cleaned_metadata = {}
        for k, v in doc.metadata.items():
            if isinstance(v, float) and math.isnan(v):
                cleaned_metadata[k] = None
            else:
                cleaned_metadata[k] = v
        pages_chunked_cleaned.append(
            Document(page_content=cleaned_text, metadata=cleaned_metadata)
        )

print(f"Number of text chunks after filtering: {len(pages_chunked_cleaned)}")

print(pages_chunked_cleaned[0])

page_content='Aurélien Géron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2019-05-07T15:51:31+00:00', 'moddate': '2019-05-07T15:51:31+00:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './AI_Book.pdf', 'total_pages': 510, 'page': 2, 'page_label': '1'}
Number of text chunks after filtering: 1258
page_content='Aurlien Gron
Handson Machine Learning with
ScikitLearn Keras and
TensorFlow
Concepts Tools and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Out

In [29]:
print(pages_chunked_cleaned[1])

page_content='9781492032649
LSI
Handson Machine Learning with ScikitLearn Keras and TensorFlow
by Aurlien Gron
Copyright  2019 Aurlien Gron All rights reserved
Printed in the United States of America
Published by OReilly Media Inc 1005 Gravenstein Highway North Sebastopol CA 95472
OReilly books may be purchased for educational business or sales promotional use Online editions are
also available for most titles httporeillycom For more information contact our corporateinstitutional
sales department 8009989938 or corporateoreillycom
Editor Nicole Tache
Interior Designer David Futato
Cover Designer Karen Montgomery
Illustrator Rebecca Demarest
June 2019  Second Edition
Revision History for the Early Release
20181105 First Release
20190124 Second Release
20190307 Third Release
20190329 Fourth Release
20190422 Fifth Release
See httporeillycomcatalogerratacspisbn9781492032649 for release details' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Fo

In [30]:
print(f"Number of text chunks: {len(pages_chunked_cleaned)}")

Number of text chunks: 1258


#### Task 3: Store vector embeddings from pdf document to ChromaDB vector database.

**Objective:**

Store vector embeddings into ChromaDB to store knowledge.

**Task Description:**

- create chromadb client
- create chromadb collection
- create langchain chroma db client
- store text document chunks and vector embeddings to vector databases

**Useful links:**

- [Langchain How To](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#initialization-from-client)

In [31]:
from langchain_chroma import Chroma
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

collection_name = "ai_model_book"

# ADD HERE YOUR CODE
# Create a collection
collection = client.get_or_create_collection(name=collection_name)

# ADD HERE YOUR CODE
# Create chromadb
vector_db_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model
)

In [32]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(pages_chunked_cleaned))]

# ADD HERE YOUR CODE
vector_db_from_client.add_documents(documents=pages_chunked_cleaned, ids=uuids)

['b55aa98f-2c2e-4ff9-9b4a-5829893dd432',
 'dd7ff243-ea3d-499a-a14a-c94a41d7c913',
 'b313d29c-8f89-42db-9ea2-cf2a82d3ab4b',
 'ca3b61d7-105f-4714-8829-014b01d683eb',
 'b85b7435-ed63-4133-870f-cf124abf92a3',
 '0292cab0-7c22-4a2a-bdae-a49e7929e2b7',
 '26eeae64-d35a-4850-805f-f7b4f8ade24c',
 '0a9c576f-429e-4be3-868c-5d4d852a29de',
 '50769665-2678-4d25-bea8-6f5ac9f0f391',
 'cfe84682-af63-4709-95ac-d4892a3dd448',
 '021ee521-8369-4c0f-81d0-0092a1bce19a',
 '5561783c-0ea7-4495-b8f8-8e01752ab271',
 '137f2cc8-49fb-4e88-88e8-0a6e355622e5',
 'a01e0d9c-5626-4eaa-9b0d-76049d9cc546',
 '6158f8ab-c743-48a8-980f-a2c29d03595a',
 'c79adaee-644d-43af-94cd-4dd3702e1b6c',
 '16dd13b1-b544-45f8-8dd9-0944832aff8c',
 '8bd8e6e5-c942-48a4-b289-1632b6d84c25',
 'da232d66-db59-46de-9a4c-b6d6cbdc7fc6',
 'fcc96892-0c74-4a3c-ab50-349c3c429dbe',
 '444423f4-da4e-4931-9036-646db7ec392d',
 '05e930d6-84ce-49bf-9ba2-148161129396',
 '4f98ebe4-7532-4c38-91c0-105f763b9d4b',
 'f50d9708-5ffd-43a6-b99e-45a096279a40',
 '5643ba68-7f4b-

In [33]:
client.count_collections()

1

In [34]:
# client.delete_collection("ai_model_book")

In [35]:
# Get and check the content of the collection
collection.get(include=["documents", "metadatas", "embeddings"])

{'ids': ['b55aa98f-2c2e-4ff9-9b4a-5829893dd432',
  'dd7ff243-ea3d-499a-a14a-c94a41d7c913',
  'b313d29c-8f89-42db-9ea2-cf2a82d3ab4b',
  'ca3b61d7-105f-4714-8829-014b01d683eb',
  'b85b7435-ed63-4133-870f-cf124abf92a3',
  '0292cab0-7c22-4a2a-bdae-a49e7929e2b7',
  '26eeae64-d35a-4850-805f-f7b4f8ade24c',
  '0a9c576f-429e-4be3-868c-5d4d852a29de',
  '50769665-2678-4d25-bea8-6f5ac9f0f391',
  'cfe84682-af63-4709-95ac-d4892a3dd448',
  '021ee521-8369-4c0f-81d0-0092a1bce19a',
  '5561783c-0ea7-4495-b8f8-8e01752ab271',
  '137f2cc8-49fb-4e88-88e8-0a6e355622e5',
  'a01e0d9c-5626-4eaa-9b0d-76049d9cc546',
  '6158f8ab-c743-48a8-980f-a2c29d03595a',
  'c79adaee-644d-43af-94cd-4dd3702e1b6c',
  '16dd13b1-b544-45f8-8dd9-0944832aff8c',
  '8bd8e6e5-c942-48a4-b289-1632b6d84c25',
  'da232d66-db59-46de-9a4c-b6d6cbdc7fc6',
  'fcc96892-0c74-4a3c-ab50-349c3c429dbe',
  '444423f4-da4e-4931-9036-646db7ec392d',
  '05e930d6-84ce-49bf-9ba2-148161129396',
  '4f98ebe4-7532-4c38-91c0-105f763b9d4b',
  'f50d9708-5ffd-43a6-b99e-

#### Task 4: Access ChromaDB and perform vector search

**Objective:**

Use query to perform vector search against ChromaDB vector database

**Task Description:**

- define query
- run vector search
- print k=3 most similar documents


**Useful links:**

- [Langchain Query ChromaDB](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#search-by-vector)

In [ ]:
search_query = "Types of Machine Learning Systems"

results = vector_db_from_client.similarity_search(search_query, k=3)

for res in results:
    print(res.page_content)
    print(res.metadata)
    print("\n")

connections to recover some of the spatial resolution that is lost in the CNN we will
discuss this shortly when we look at semantic segmentation Moreover in the 2016
paper the authors introduce the YOLO9000 model that uses hierarchical classifica
tion the model predicts a probability for each node in a visual hierarchy called Word
Tree This makes it possible for the network to predict with high confidence that an
image represents say a dog even though it is unsure what specific type of dog it is
476  Chapter 14 Deep Computer Vision Using Convolutional Neural Networks
{'source': './AI_Book.pdf', 'trapped': '/False', 'creationdate': '2019-05-07T15:51:31+00:00', 'page_label': '476', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'moddate': '2019-05-07T15:51:31+00:00', 'page': 501, 'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'total_pages': 510, 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}


the classif